# Load dcm file

In [1]:
import numpy as np
import pydicom
import os


def load_dicom_images(dicom_dir):
    # Get list of DICOM files in the directory
    dicom_files = sorted([os.path.join(dicom_dir, file) for file in os.listdir(dicom_dir) if file.endswith('.dcm')])
    try:
        # Read DICOM files and stack pixel arrays into a 3D array
        dicom_data = [pydicom.dcmread(file) for file in dicom_files]
        preprocessed_image_data = np.stack([dicom.pixel_array for dicom in dicom_data])
        # Remove single channel dimension if present
        preprocessed_image_data = np.squeeze(preprocessed_image_data)
        return preprocessed_image_data
    except Exception as e:
        print("Error loading DICOM data:", e)
        return None


# Code for saving the slices

In [12]:
import os
import numpy as np
import matplotlib.pyplot as plt

def save_slices(output_dir, patient_id, datscan_date, dicom_images):
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    print("output_dir:", output_dir)

    # Save axial slices 37 to 44 as PNG
    for slice_number in range(37, 45):
        output_filename = f"{patient_id}_{datscan_date}_{slice_number}.png"       
        output_path = os.path.join(output_dir, output_filename)
        try:
            if 0 <= slice_number < dicom_images.shape[0]:
                plt.imshow(dicom_images[slice_number, :, :], cmap='gray')
                plt.axis('off')
                
                # Check if file already exists
                if os.path.exists(output_path):
                    os.remove(output_path)
                    print(f"File {output_filename} already exists. Overwriting...")

                plt.savefig(output_path, bbox_inches='tight', pad_inches=0)
                plt.close()
                print(f"Saved axial slice {slice_number} as PNG:", output_path)
            else:
                print(f"Skipping out of bounds slice {slice_number} for Patient ID {patient_id}, DATSCAN Date {datscan_date}")
                break
        except IndexError:
            print(f"IndexError: Skipping DICOM images for Patient ID {patient_id}, DATSCAN Date {datscan_date}")
            break
        except Exception as e:
            print(f"Error saving axial slice {slice_number} for Patient ID {patient_id}, DATSCAN Date {datscan_date}: {e}")
            break
    else:
        print("All slices saved successfully.")


# Code for travers through the directories and save the slices

In [13]:
def traverse_directory(root_dir, output_root_dir):
        print("Starting directory traversal...")
        for patient_id in os.listdir(root_dir):
            patient_dir = os.path.join(root_dir, patient_id)
            output_patient_dir = os.path.join(output_root_dir, patient_id)
            if os.path.isdir(patient_dir):
                    #  Inside each patient directory, look for Reconstructed_DaTSCAN directory
                    reconstructed_datscan_dir = os.path.join(patient_dir, "Reconstructed_DaTSCAN")
                    if os.path.isdir(reconstructed_datscan_dir):
                        # Inside Reconstructed_DaTSCAN, look for date folders
                        for date_folder in os.listdir(reconstructed_datscan_dir):
                            date_folder_path = os.path.join(reconstructed_datscan_dir, date_folder)
                            if os.path.isdir(date_folder_path):
                                # Inside each date folder, look for DATSCAN ID folders
                                for datscan_id_folder in os.listdir(date_folder_path):
                                    datscan_id_folder_path = os.path.join(date_folder_path, datscan_id_folder)
                                    if os.path.isdir(datscan_id_folder_path):
                                        # Inside each DATSCAN ID folder, load DICOM images and save axial slices
                                        dicom_images = load_dicom_images(datscan_id_folder_path)
                                        if dicom_images is not None:
                                            save_slices(output_root_dir, patient_id, date_folder, dicom_images) 

# Call the functions

In [14]:
#output folder path will be patient id/date/slice number.png
root_dir = r"/home/m8m/Projects/PPMI_Research_on_Parkinsons/src/study_patient/PPMI"
# Output directory to save PNG files for pd and hc groups
output_root = r"/home/m8m/Projects/PPMI_Research_on_Parkinsons/src/review/slices_output"
traverse_directory(root_dir, output_root)
print('done')

Starting directory traversal...
output_dir: /home/m8m/Projects/PPMI_Research_on_Parkinsons/src/review/slices_output
Saved axial slice 37 as PNG: /home/m8m/Projects/PPMI_Research_on_Parkinsons/src/review/slices_output/3532_2016-04-20_15_04_49.0_37.png
Saved axial slice 38 as PNG: /home/m8m/Projects/PPMI_Research_on_Parkinsons/src/review/slices_output/3532_2016-04-20_15_04_49.0_38.png
Saved axial slice 39 as PNG: /home/m8m/Projects/PPMI_Research_on_Parkinsons/src/review/slices_output/3532_2016-04-20_15_04_49.0_39.png
Saved axial slice 40 as PNG: /home/m8m/Projects/PPMI_Research_on_Parkinsons/src/review/slices_output/3532_2016-04-20_15_04_49.0_40.png
Saved axial slice 41 as PNG: /home/m8m/Projects/PPMI_Research_on_Parkinsons/src/review/slices_output/3532_2016-04-20_15_04_49.0_41.png
Saved axial slice 42 as PNG: /home/m8m/Projects/PPMI_Research_on_Parkinsons/src/review/slices_output/3532_2016-04-20_15_04_49.0_42.png
Saved axial slice 43 as PNG: /home/m8m/Projects/PPMI_Research_on_Parkinson